# fixed_effects_ntv

> 对应代码：`EconometricsCode/code_in_notes/Chap.13/fixed_effects_ntv.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.13`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.13")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们先做环境设置并读入 NTV 事件的地区—年度面板数据，用 `xtset` 声明面板维度：个体为选区（tik_id），时间为年份（year）。声明面板结构后，Stata 才能识别 `L.`、`L4.` 等滞后算子。

In [ ]:
%%stata
clear
set more off
use ../datasets/NTV_Aggregate_Data_reshaped.dta
xtset tik_id year

为了灵活地控制社会经济条件的非线性影响，我们生成滞后人口对数与滞后工资对数的 1 至 5 次多项式项。`forvalues` 循环依次生成 `log_pop_1`…`log_pop_5` 和 `log_wage_1`…`log_wage_5`，稍后以通配符 `L.log_pop_*` 一次性放入回归。

In [ ]:
%%stata
forvalues i=1/5{
    gen log_pop_`i'=logpop^`i'
    gen log_wage_`i'=log_wage^`i'
}

第一个设定是最简单的基准回归：以 1999 年“统一俄罗斯”的得票率为被解释变量，核心解释变量是 NTV 收视率（Watch_probit）。我们用 `reghdfe` 吸收地区固定效应（`absorb(region)`），并在地区层面聚类标准误（`cluster(region)`），以允许同一地区内部误差项相关。这一步考察的是：在只控制地区异质性时，NTV 曝光与投票行为之间的相关关系。

In [ ]:
%%stata
// 简单回归
reghdfe Votes_Edinstvo_ Watch_probit if year==1999, absorb(region)  cluster(region)

接下来我们加入社会经济控制变量。把滞后的人口、工资（含各次幂）以及护士数、医生数存入局部宏 `E_controls`，然后在回归中整体引用。这些控制变量缓解“收视率高的地区本身社会经济特征不同”所带来的遗漏变量偏误。

In [ ]:
%%stata
// 加入社会经济控制
local E_controls "L.log_pop_* L.log_wage_* L.nurses L.doctors_pc"
reghdfe Votes_Edinstvo_ Watch_probit `E_controls' if year==1999, absorb(region) cluster(region)

最后我们进一步控制 1995 年（即 4 期滞后，故用 `L4.`）各主要党派的得票率与投票率，存入 `X_controls`。这一步控制了选区长期以来的政治倾向，使识别更接近“在既往政治格局相似的选区之间，比较 NTV 曝光差异对 1999 年统一俄罗斯得票的影响”。三个设定并列，可以观察核心系数对控制变量逐步丰富的稳健性。

In [ ]:
%%stata
// 加入1995年选举的变量
local X_controls "L4.Votes_KPRF_ L4.Votes_Yabloko_ L4.Votes_NDR_ L4.Votes_LDPR_ L4.Turnout"
reghdfe Votes_Edinstvo_ Watch_probit `E_controls' `X_controls' if year==1999, absorb(region) cluster(region)